# Tech Challenge Fase 3 — Construção da Gold de Modelagem

## Objetivo

Construir a tabela individual que será utilizada nas etapas de EDA e Machine Learning:

`workspace.alfabetizacao_gold.base_modelagem_aluno`

A granularidade será:

> **1 linha = 1 aluno em determinado ano**

A tabela combina:

- dados educacionais da `fato_alunos`;
- informações territoriais;
- variáveis socioeconômicas municipais;
- variável-alvo `alfabetizado`;
- identificação do grupo metodológico de treino e teste.

### Decisões importantes

A variável `proficiencia` não será incluída porque o diagnóstico mostrou relação praticamente determinística com o target, caracterizando risco de **data leakage**.

Também não serão utilizadas nesta base de modelagem:

- `serie`, por possuir variância zero;
- `peso_aluno`, por representar peso amostral;
- `caderno` e `preenchimento_caderno`, por serem variáveis operacionais da avaliação;
- `id_escola`, por ser um identificador de alta cardinalidade.

Os identificadores `id_aluno` e `id_municipio` serão preservados para controle metodológico e análises, mas não serão necessariamente utilizados como features do modelo.


## 1. Configurações


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER_SCHEMA = "alfabetizacao_silver"
GOLD_SCHEMA = "alfabetizacao_gold"

ALUNOS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.fato_alunos"
)

SOCIO_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.socioeconomico_municipio"
)

OUTPUT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.base_modelagem_aluno"
)

print("Alunos:", ALUNOS_TABLE)
print("Socioeconômico:", SOCIO_TABLE)
print("Destino:", OUTPUT_TABLE)


## 2. Leitura das fontes


In [ ]:
df_alunos = spark.table(ALUNOS_TABLE)
df_socio = spark.table(SOCIO_TABLE)

print("Registros fato_alunos:", df_alunos.count())
print("Registros socioeconomico_municipio:", df_socio.count())

print("\nColunas fato_alunos:")
print(df_alunos.columns)

print("\nColunas socioeconômicas:")
print(df_socio.columns)


## 3. Seleção das variáveis individuais

Nesta etapa são selecionadas apenas as variáveis necessárias para identificação, contexto educacional, contexto territorial e target.

A variável `proficiencia` é propositalmente excluída.


In [ ]:
df_alunos_modelagem = (
    df_alunos
    .select(
        F.col("ano"),
        F.col("id_aluno"),
        F.col("id_municipio"),
        F.col("nome_municipio"),
        F.col("sigla_uf"),
        F.col("rede"),
        F.col("presenca"),
        F.col("alfabetizado"),
    )
)

display(df_alunos_modelagem.limit(10))


## 4. Enriquecimento socioeconômico

A tabela socioeconômica possui aproximadamente uma linha por município e ano.

Como ela é pequena em relação à base de alunos, é utilizado `broadcast join` para tornar a integração mais eficiente.


In [ ]:
df_base = (
    df_alunos_modelagem.alias("a")
    .join(
        F.broadcast(
            df_socio.select(
                "ano",
                "id_municipio",
                "populacao",
                "ano_rais",
                "quantidade_vinculos_ativos",
                "quantidade_vinculos_clt",
                "quantidade_vinculos_estatutarios",
                "vinculos_ativos_por_1000_habitantes",
            )
        ).alias("s"),
        on=["ano", "id_municipio"],
        how="left",
    )
)

display(df_base.limit(10))


## 5. Definição dos grupos metodológicos

O projeto utilizará uma separação temporal simples e reproduzível:

- **DESENVOLVIMENTO_2023**: registros de 2023, utilizados em treinamento e cross-validation;
- **TESTE_2024_NOVO**: alunos de 2024 que não aparecem em 2023, reservados para o teste final;
- **RESERVA_2024_RECORRENTE**: alunos de 2024 já observados em 2023, mantidos na Gold mas fora do teste final principal.

Essa estratégia permite avaliar o modelo em alunos temporalmente posteriores e individualmente não observados durante o treinamento.


In [ ]:
ids_2023 = (
    df_alunos_modelagem
    .filter(F.col("ano") == 2023)
    .select("id_aluno")
    .distinct()
    .withColumn(
        "_presente_2023",
        F.lit(1),
    )
)

df_base = (
    df_base
    .join(
        ids_2023,
        on="id_aluno",
        how="left",
    )
    .withColumn(
        "grupo_modelagem",
        F.when(
            F.col("ano") == 2023,
            F.lit("DESENVOLVIMENTO_2023"),
        )
        .when(
            (F.col("ano") == 2024)
            & F.col("_presente_2023").isNull(),
            F.lit("TESTE_2024_NOVO"),
        )
        .otherwise(
            F.lit("RESERVA_2024_RECORRENTE"),
        ),
    )
    .drop("_presente_2023")
)


## 6. Preparação do target

Além do target booleano original, será criada uma representação numérica:

- `1` = alfabetizado;
- `0` = não alfabetizado.

A conversão facilita o uso posterior no Scikit-learn.


In [ ]:
df_base = (
    df_base
    .withColumn(
        "target_alfabetizado",
        F.when(
            F.col("alfabetizado") == True,
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        "_gold_processed_at",
        F.current_timestamp(),
    )
)


## 7. Organização final das colunas


In [ ]:
df_base = (
    df_base
    .select(
        # Identificação
        "id_aluno",
        "ano",

        # Territoriais
        "id_municipio",
        "nome_municipio",
        "sigla_uf",

        # Educacionais
        "rede",
        "presenca",

        # Socioeconômicas
        "populacao",
        "ano_rais",
        "quantidade_vinculos_ativos",
        "quantidade_vinculos_clt",
        "quantidade_vinculos_estatutarios",
        "vinculos_ativos_por_1000_habitantes",

        # Controle metodológico
        "grupo_modelagem",

        # Target
        "alfabetizado",
        "target_alfabetizado",

        # Metadado técnico
        "_gold_processed_at",
    )
)

display(df_base.limit(20))


## 8. Validação de volume e grupos

As contagens devem preservar os registros da base individual e reproduzir a estratégia definida no diagnóstico.


In [ ]:
display(
    df_base
    .groupBy(
        "ano",
        "grupo_modelagem",
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("id_aluno").alias("alunos_distintos"),
        F.sum("target_alfabetizado").alias("alfabetizados"),
        F.round(
            F.avg("target_alfabetizado") * 100,
            2,
        ).alias("percentual_alfabetizados"),
    )
    .orderBy(
        "ano",
        "grupo_modelagem",
    )
)


## 9. Validação de completude

Como o Notebook 01 apresentou 100% de cobertura territorial, esperamos ausência de nulos nas variáveis socioeconômicas principais.


In [ ]:
display(
    df_base
    .groupBy("ano")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("populacao").isNull(), 1).otherwise(0)
        ).alias("populacao_nula"),
        F.sum(
            F.when(
                F.col("quantidade_vinculos_ativos").isNull(),
                1,
            ).otherwise(0)
        ).alias("vinculos_ativos_nulo"),
        F.sum(
            F.when(
                F.col("vinculos_ativos_por_1000_habitantes").isNull(),
                1,
            ).otherwise(0)
        ).alias("indicador_rais_nulo"),
        F.sum(
            F.when(F.col("alfabetizado").isNull(), 1).otherwise(0)
        ).alias("target_nulo"),
    )
    .orderBy("ano")
)


## 10. Validação de unicidade

A combinação `id_aluno + ano` deve continuar única após o enriquecimento.


In [ ]:
duplicados = (
    df_base
    .groupBy(
        "id_aluno",
        "ano",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Combinações duplicadas id_aluno + ano:",
    duplicados,
)

if duplicados != 0:
    raise ValueError(
        "Foram encontradas duplicidades após o enriquecimento."
    )


## 11. Validação do conjunto de teste

O resultado esperado é aproximadamente:

- 604.889 alunos em `TESTE_2024_NOVO`;
- target próximo de 50% / 50%.


In [ ]:
display(
    df_base
    .filter(
        F.col("grupo_modelagem") == "TESTE_2024_NOVO"
    )
    .agg(
        F.count("*").alias("alunos_teste"),
        F.sum("target_alfabetizado").alias("alfabetizados"),
        F.sum(
            1 - F.col("target_alfabetizado")
        ).alias("nao_alfabetizados"),
        F.round(
            F.avg("target_alfabetizado") * 100,
            2,
        ).alias("percentual_alfabetizados"),
    )
)


## 12. Persistência na camada Gold

Após as validações, a base individual de modelagem é persistida em Delta Lake.


In [ ]:
(
    df_base
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print(
    f"Tabela criada com sucesso: {OUTPUT_TABLE}"
)

print(
    "Quantidade de registros:",
    spark.table(OUTPUT_TABLE).count(),
)


## 13. Visão final da Gold


In [ ]:
display(
    spark.sql(
        f"""
        SELECT
            ano,
            grupo_modelagem,
            COUNT(*) AS registros,
            COUNT(DISTINCT id_municipio) AS municipios,
            COUNT(DISTINCT sigla_uf) AS ufs,
            ROUND(
                AVG(target_alfabetizado) * 100,
                2
            ) AS percentual_alfabetizados,
            ROUND(
                AVG(populacao),
                2
            ) AS populacao_media,
            ROUND(
                AVG(vinculos_ativos_por_1000_habitantes),
                2
            ) AS media_vinculos_por_1000
        FROM {OUTPUT_TABLE}
        GROUP BY
            ano,
            grupo_modelagem
        ORDER BY
            ano,
            grupo_modelagem
        """
    )
)


## Conclusão

A tabela Gold `base_modelagem_aluno` consolida em uma única visão:

- variável-alvo individual de alfabetização;
- atributos educacionais;
- atributos territoriais;
- atributos socioeconômicos;
- separação metodológica entre desenvolvimento e teste.

### Estratégia definida

**Desenvolvimento**
- dados de 2023;
- treinamento e cross-validation.

**Teste final**
- alunos de 2024 que não aparecem em 2023.

**Reserva**
- alunos de 2024 que também aparecem em 2023.

### Variáveis excluídas por decisão analítica

- `proficiencia`: risco direto de data leakage;
- `serie`: variância zero;
- `peso_aluno`: peso amostral;
- `caderno`: variável operacional;
- `preenchimento_caderno`: variável operacional;
- `id_escola`: identificador de alta cardinalidade.

O próximo passo será realizar a **Análise Exploratória de Dados (EDA)** sobre a Gold e avaliar quais features seguirão para a pipeline de Machine Learning.
